# Assignment 02: Statistical NLP Fundamentals

**Name:** Yominesh Giri

## Exercise 1: Word Frequency Counter

#### Objective : Count how many times each word appears in a sentence



In [ ]:
import re

input_text = "I like NLP I like Python NLP is fun"

# convert to lower case
input_text = input_text.lower()

#clean sentence i.e. remove punctuation
input_text=re.sub(r"'s\b|[^\w\s]", "", input_text)

#split into words
words=input_text.split()

#Count frequency of words
word_freq={}

for word in words:
  if word in word_freq:
    word_freq[word]+= 1
  else:
    word_freq[word]=1

print (word_freq)


{'i': 2, 'like': 2, 'nlp': 2, 'python': 1, 'is': 1, 'fun': 1}


## Exercise 2: Vocabulary Builder

#### Objective : Find all unique words in a corpus.

In [ ]:
corpus = [
"I like NLP",
"I like AI",
"AI is amazing"
]

unique_words = set()

for doc in corpus:
    words = re.sub(r"'s\b|[^\w\s]", "", doc.lower()).split()
    unique_words.update(words)

print(unique_words)
print (len(unique_words))

{'amazing', 'like', 'is', 'ai', 'i', 'nlp'}
6


## Exercise 3: Build a Word Co-occurrence Matrix
#### Objective : Construct a simple co-occurrence matrix using a window size of 1.

In [ ]:
sentence = "the cat sat on the mat"
words = sentence.split()

# vocabulary
vocab = sorted(set(words))

# create matrix
co_matrix = {word: {other: 0 for other in vocab} for word in vocab}

#calculation
window_size = 1
for i, word in enumerate(words):
    for j in range(max(0, i - window_size), min(len(words), i + window_size + 1)):
        if i != j:
            neighbor = words[j]
            co_matrix[word][neighbor] += 1

print("     ", "  ".join(vocab))
for word in vocab:
    row = [str(co_matrix[word][other]) for other in vocab]
    print(f"{word:5}", "  ".join(row))

      cat  mat  on  sat  the
cat   0  0  0  1  1
mat   0  0  0  0  1
on    0  0  0  1  1
sat   1  0  1  0  0
the   1  1  1  0  0


## Exercise 4: Detect Data Sparsity
#### Objective : Find word pairs that never occur together.

In [ ]:
# pairs that never co-occur
sparse_pairs = []

for word in vocab:
    for other in vocab:
        if word != other and co_matrix[word][other] == 0:
            # sort so (x,y) and (y,x) aren't both added
            pair = tuple(sorted([word, other]))
            if pair not in sparse_pairs:
                sparse_pairs.append(pair)

print("Word pairs that never occur together:")
for pair in sparse_pairs:
    print(pair)

Word pairs that never occur together:
('cat', 'mat')
('cat', 'on')
('mat', 'on')
('mat', 'sat')
('sat', 'the')


## Exercise 5: Implement Laplace Smoothing
#### Objective : Calculate word probabilities using Laplace smoothing.

In [ ]:
word_counts = {"cat": 4, "dog": 3, "bird": 1}

# total count of all words
total = sum(word_counts.values())

#vocabulary size
vocab = len(word_counts)

# apply Laplace smoothing formula to each word
smoothed_prob = {}
for word, count in word_counts.items():
    smoothed_prob[word] = (count + 1) / (total + vocab)

for word, prob in smoothed_prob.items():
    print(f"{word} : {prob:.2f}")

cat : 0.45
dog : 0.36
bird : 0.18


## Exercise 6: Simple Keyboard Prediction
#### Objective : Predict the most common next word.

In [ ]:
corpus = [
    "I like NLP",
    "I like Python",
    "I like coffee",
    "I love NLP"
]

# build bigram counts
bigram_counts = {}

for sentence in corpus:
    words = sentence.lower().split()
    for i in range(len(words) - 1):
        current_word = words[i]
        next_word = words[i + 1]

        if current_word not in bigram_counts:
            bigram_counts[current_word] = {}

        bigram_counts[current_word][next_word] = bigram_counts[current_word].get(next_word, 0) + 1

print("Bigram counts:", bigram_counts)

# find next-word frequencies from given input
input_text = "i like"
last_word = input_text.lower().split()[-1]

next_word_freq = bigram_counts.get(last_word, {})
print("\nNext word frequencies for '", last_word, "':", next_word_freq)

# Convert frequencies into probabilities
total = sum(next_word_freq.values())
probabilities = {word: count / total for word, count in next_word_freq.items()}

# Sort and display top 3 predictions
top_3 = sorted(probabilities.items(), key=lambda x: x[1], reverse=True)[:3]

print("\nTop 3 predictions after 'I like':")
for word, prob in top_3:
    print(f"{word} : {prob:.1f}")

Bigram counts: {'i': {'like': 3, 'love': 1}, 'like': {'nlp': 1, 'python': 1, 'coffee': 1}, 'love': {'nlp': 1}}

Next word frequencies for ' like ': {'nlp': 1, 'python': 1, 'coffee': 1}

Top 3 predictions after 'I like':
nlp : 0.3
python : 0.3
coffee : 0.3


## Exercise 7: Calculate TF-IDF
#### Objective : Compute TF, IDF, and TF-IDF scores.

In [ ]:
import math

corpus = [
    "I like NLP",
    "I like AI",
    "AI is amazing"
]

# tokenize each document
docs = [doc.lower().split() for doc in corpus]

# build vocabulary
vocab = set()
for doc in docs:
    vocab.update(doc)

# compute Term Frequency (TF) per document
# TF = (count of word in doc) / (total words in doc)
def compute_tf(doc):
    tf = {}
    total_words = len(doc)
    for word in doc:
        tf[word] = tf.get(word, 0) + 1
    for word in tf:
        tf[word] = tf[word] / total_words
    return tf

tf_per_doc = [compute_tf(doc) for doc in docs]

# compute Inverse Document Frequency (IDF)
# IDF = log(total number of docs / number of docs containing the word)
def compute_idf(docs, vocab):
    idf = {}
    total_docs = len(docs)
    for word in vocab:
        doc_count = sum(1 for doc in docs if word in doc)
        idf[word] = math.log(total_docs / doc_count)
    return idf

idf = compute_idf(docs, vocab)

# compute TF-IDF = TF * IDF for each document
tfidf_per_doc = []
for tf in tf_per_doc:
    tfidf = {word: tf[word] * idf[word] for word in tf}
    tfidf_per_doc.append(tfidf)

for i, doc in enumerate(corpus):
    print(f"\nDocument {i+1}: \"{doc}\"")
    print("TF:", tf_per_doc[i])
    print("TF-IDF:", {word: round(score, 3) for word, score in tfidf_per_doc[i].items()})

print("\nIDF (whole corpus):", {word: round(score, 3) for word, score in idf.items()})


Document 1: "I like NLP"
TF: {'i': 0.3333333333333333, 'like': 0.3333333333333333, 'nlp': 0.3333333333333333}
TF-IDF: {'i': 0.135, 'like': 0.135, 'nlp': 0.366}

Document 2: "I like AI"
TF: {'i': 0.3333333333333333, 'like': 0.3333333333333333, 'ai': 0.3333333333333333}
TF-IDF: {'i': 0.135, 'like': 0.135, 'ai': 0.135}

Document 3: "AI is amazing"
TF: {'ai': 0.3333333333333333, 'is': 0.3333333333333333, 'amazing': 0.3333333333333333}
TF-IDF: {'ai': 0.135, 'is': 0.366, 'amazing': 0.366}

IDF (whole corpus): {'i': 0.405, 'like': 0.405, 'is': 1.099, 'nlp': 1.099, 'amazing': 1.099, 'ai': 0.405}
